In [25]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from rich import print as rprint
from rich.pretty import pprint

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [26]:
# Tools and Schemas

from datetime import datetime, timedelta
from anthropic.types import ToolParam


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def batch_tool(invocations = []):
    tool_results = []

    for func_obj in invocations:
        tool_results.append(TOOL_MAP[func_obj.name](**func_obj.arguments))
    
    return tool_results

add_duration_to_datetime_schema: ToolParam = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema: ToolParam = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema: ToolParam = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

get_current_datetime_schema: ToolParam = {
  "name": "get_current_datetime",
  "description": "Returns the current date and time formatted according to the specified format string.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A strftime-compatible format string (e.g. '%Y-%m-%d %H:%M:%S'). Cannot be empty.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
}

TOOL_MAP = {
    "add_duration_to_datetime": add_duration_to_datetime,
    "set_reminder": set_reminder,
    "get_current_datetime": get_current_datetime
}

In [27]:
# Helper functions
def add_user_message(messages, content):
    user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    assistant_message = {"role": "assistant", "content": content}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
        "tools": tools,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def run_tools(messages, content):
    for block in content:
        if block.type == "tool_use":
            try:
                tool_result = TOOL_MAP[block.name](**block.input)

                add_user_message(messages=messages, content=[{
                    "tool_use_id": block.id,
                    "type": "tool_result",
                    "content": tool_result,
                    "is_error": False
                }])
            except Exception as e:
                add_user_message(messages=messages, content=[{
                    "tool_use_id": block.id,
                    "type": "tool_result",
                    "content": str(e),
                    "is_error": True
                }])

In [28]:
messages = []
user_prompt = input("Enter prompt")
add_user_message(messages=messages, content=[{
    "type": "text",
    "text": user_prompt
}])

def run_conversation(messages):
    try:
        while True:
            response = chat(messages=messages, 
                        tools=[batch_tool_schema, get_current_datetime_schema, add_duration_to_datetime_schema])
            
            add_assistant_message(messages, response.content)
            
            if response.stop_reason == "end_turn":
                break

            if response.stop_reason == "tool_use":
                run_tools(messages=messages, content=response.content)
    except Exception as e:
        print(e)
    
    return messages

pprint(run_conversation(messages))

[
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {'type': 'text', 'text': "what's current datetime in SS format? Also, show datetime in HH:MM format"}
│   │   ]
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_013qMMRLN61pszqRW27PAPFZ',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={
│   │   │   │   │   'invocations': [
│   │   │   │   │   │   {'name': 'get_current_datetime', 'arguments': '{"date_format": "%S"}'},
│   │   │   │   │   │   {'name': 'get_current_datetime', 'arguments': '{"date_format": "%H:%M"}'}
│   │   │   │   │   ]
│   │   │   │   },
│   │   │   │   name='batch_tool',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'tool_use_id': 'toolu_013qMMRLN61pszqRW27PAPFZ',
│   │   │   │   'type': 'tool_result',
│   │   │   │   'content': "'batch_tool'",
│   │   │   │   'is_error': True
│   │   │   }
│   │   ]
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(citations=None, text='Let me try calling these separately:', type='text'),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01P22MoTCjkDTtSUzqD68CvL',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={'date_format': '%S'},
│   │   │   │   name='get_current_datetime',
│   │   │   │   type='tool_use'
│   │   │   ),
│   │   │   ToolUseBlock(
│   │   │   │   id='toolu_01MebrWsdSgtEuYhx8truU1R',
│   │   │   │   caller=DirectCaller(type='direct'),
│   │   │   │   input={'date_format': '%H:%M'},
│   │   │   │   name='get_current_datetime',
│   │   │   │   type='tool_use'
│   │   │   )
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'tool_use_id': 'toolu_01P22MoTCjkDTtSUzqD68CvL',
│   │   │   │   'type': 'tool_result',
│   │   │   │   'content': '10',
│   │   │   │   'is_error': False
│   │   │   }
│   │   ]
│   },
│   {
│   │   'role': 'user',
│   │   'content': [
│   │   │   {
│   │   │   │   'tool_use_id': 'toolu_01MebrWsdSgtEuYhx8truU1R',
│   │   │   │   'type': 'tool_result',
│   │   │   │   'content': '16:49',
│   │   │   │   'is_error': False
│   │   │   }
│   │   ]
│   },
│   {
│   │   'role': 'assistant',
│   │   'content': [
│   │   │   TextBlock(
│   │   │   │   citations=None,
│   │   │   │   text='Here are the results:\n\n- **Current seconds (SS format):** 10\n- **Current time (HH:MM format):** 16:49',
│   │   │   │   type='text'
│   │   │   )
│   │   ]
│   }
]

In [29]:
# implement batch tool calling by yourself